In [ ]:
import matplotlib.pyplot as plt

# Daten für TensorRT-Modelle
models = [
    "BEVDetNet FP32 TRT", "BEVDetNet FP16 TRT", "BEVDetNet INT8 TRT",
    "Our FP32 TRT", "Our FP16 TRT", "Our INT8 TRT"
]
latency = [6, 4, 2.1, 2.9, 1.7, 1.5]
performance = [87.51, 86.44, 84.20, 76.52, 76.50, 67.32]
colors = ['blue', 'blue', 'blue', 'red', 'red', 'red']
sizes = [59, 39, 22, 49, 25, 16]  # model size in MB

# Diagramm mit größerer Figur erstellen
plt.figure(figsize=(12, 7))
scatter = plt.scatter(latency, performance, c=colors, s=[s*10 for s in sizes], alpha=0.7)

# Achsenbereiche manuell festlegen
plt.xlim(0.8, 7.5)  # Latenz-Achse von 0.8ms bis 7.5ms
plt.ylim(60, 92)    # Performance-Achse von 60% bis 92%

# Beschriftungen mit Pfeilen
for i, model in enumerate(models):
    if i < 3:  # BEVDetNet Modelle
        offset = (15, 15)
    elif i == 3:  # Unser FP32 TRT - weiter nach rechts
        offset = (30, -20)  # Größerer x-Offset für mehr Abstand nach rechts
    else:  # Unser FP16 TRT und INT8 TRT - unverändert
        offset = (15, -20)
    
    plt.annotate(f"{model}\n{performance[i]}% | {latency[i]}ms | {sizes[i]}MB", 
                (latency[i], performance[i]), 
                xytext=offset,
                textcoords="offset points",
                ha='left',
                va='bottom' if i < 3 else 'top',
                fontsize=7,
                arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color='gray'),
                bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.8))

plt.xlabel('Latency (ms)', fontsize=12)
plt.ylabel('Average Precision (%)', fontsize=12)
plt.title('Comparison of TensorRT models: latency vs. average precision', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)

# Legende erstellen
blue_patch = plt.Line2D([0], [0], marker='o', color='w', label='BEVDetNet TRT', 
                       markerfacecolor='blue', markersize=8)
red_patch = plt.Line2D([0], [0], marker='o', color='w', label='Our model TRT', 
                      markerfacecolor='red', markersize=8)

# Größen-Legende
size_legends = []
for size, label in zip([20, 40, 60], ['Small', 'Medium', 'Large']):
    size_legends.append(plt.scatter([], [], c='gray', alpha=0.5, s=size*5, label=label))

# Legende weiter rechts platzieren (bbox_to_anchor x-Wert erhöht)
legend = plt.legend(handles=[blue_patch, red_patch] + size_legends,
                   title='Legend', loc='upper left', 
                   bbox_to_anchor=(1.18, 1),  # x-Position von 1.02 auf 1.18 erhöht
                   borderaxespad=0.,
                   handletextpad=1.5,
                   labelspacing=1.2)

# Rahmen um die Legende
legend.get_frame().set_alpha(0.8)
legend.get_frame().set_edgecolor('black')

# Anpassung des Layouts für mehr Platz rechts
plt.tight_layout(rect=[0, 0, 0.82, 1])  # rect Parameter angepasst

#plt.savefig('tensorrt_comparison.svg', format='svg', dpi=300, bbox_inches='tight')
plt.show()

## Metrics plotter with normalized (ms/TFLOPS) latency

In [ ]:
import matplotlib.pyplot as plt

# Data for TensorRT models
models = ["BEVDetNet FP32 TRT", "BEVDetNet FP16 TRT", "BEVDetNet INT8 TRT",
    "Our FP32 TRT", "Our FP16 TRT", "Our INT8 TRT"]

latency_original = [6, 4, 2.1, 2.8, 1.7, 1.5]  # BEVDETNET latency in ms
performance = [87.51, 86.44, 84.20, 86.03, 86.01, 80.67]
colors = ['blue', 'blue', 'blue', 'red', 'red', 'red']
sizes = [59, 39, 22, 49, 25, 16]  # model size in MB

# GPU FLOPS (in TFLOPS)
bevdetnet_flops = 11.34  # NVIDIA GeForce GTX 1080 Ti (11GB)
rtx3080_fp32_flops = 29.77  # NVIDIA RTX 3080 (10GB)
rtx3080_fp16_flops = 29.77  # NVIDIA RTX 3080 (10GB)
rtx3080_int8_flops = 0.4651 # NVIDIA RTX 3080 (10GB)


# Normalization of latency based on FLOPS (latency per TFLOPS)
# Formula: normalised_latency = original_latency / gpu_flops
latency_normalized = []
for i, latency in enumerate(latency_original):
    if i < 3:  # BEVDetNet models (on GTX 1080Ti)
        normalized = latency
    else:  # Our model (on RTX 3080)
        if i == 3:  # Our FP32
            normalized = latency * rtx3080_fp32_flops / bevdetnet_flops
        elif i == 4:  # Our FP16
            normalized = latency * rtx3080_fp16_flops / bevdetnet_flops
        else:   # Our INT8
            normalized = latency * rtx3080_int8_flops / bevdetnet_flops
    latency_normalized.append(normalized)

# Create a diagram with a larger figure
fig, ax1 = plt.subplots(figsize=(16, 7))

# Unnormalized latency (ms) on ax1
scatter_unnorm = ax1.scatter(latency_original, performance, c=colors, s=[s*10 for s in sizes], alpha=0.7, marker='o', label='Unnormalized latency (ms)')
ax1.set_xlabel('Latency (ms)', fontsize=18)
ax1.set_xlim(0.8, 7.5)

# Second x-axis for standardized values
ax2 = ax1.twiny()
scatter_norm = ax2.scatter(latency_normalized[3:], performance[3:], c=colors[3:], s=[s*10 for s in sizes[3:]], alpha=0.7, marker='D', label='Normalized latency (ms/TFLOPS)')
ax2.set_xlabel('Normalized latency (ms/TFLOPS)*', fontsize=18)
ax2.set_xlim(min(latency_normalized) * -1.5, max(latency_normalized) * 1.11)

ax1.set_ylabel('Average Precision (%)', fontsize=18)
ax1.set_ylim(80, 89)
ax1.grid(True, linestyle='--', alpha=0.5)

# Labels with arrows
for i, model in enumerate(models):
    if i < 3:
        offset = ()
        if i == 0:  # FP32 
            offset = (-33, -25)
        elif i == 1:  # FP16
            offset = (-33, -25)
        else:  # INT8 
            offset = (-33, -25)
    elif i == 3:  # Our FP32 TRT 
        offset = (-35, -20)
    elif i == 4:  # Our FP16 TRT
        offset = (-46, -20)
    else:  # Our INT8 TRT
        offset = (-33, -25)
    
    ax1.annotate(
        f"{model}\n{performance[i]}% | {latency_original[i]}ms | {sizes[i]}MB",
        (latency_original[i], performance[i]),
        xytext=offset,
        textcoords="offset points",
        ha='left',
        va='top',
        fontsize=7,
        arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color='gray'),
        bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.8)
    )
    
# Labels with arrows for GPU-normalized
for i, model in enumerate(models):
    if i < 3:
        continue  # BEVDetNet-Modelle NICHT auf ax2 annotieren!
    if i == 3:  # Our FP32 TRT
        offset = (-56, 40)
    elif i == 4:  # Our FP16 TRT
        offset = (-50, 40)
    else:  # Our INT8 TRT
        offset = (-5, 90)
    
    ax2.annotate(f"{model}\n{performance[i]}% | {latency_normalized[i]:.3f}ms | {sizes[i]}MB", 
                (latency_normalized[i], performance[i]), 
                xytext=offset,
                textcoords="offset points",
                ha='left',
                va='top',
                fontsize=6,
                arrowprops=dict(arrowstyle="->", connectionstyle="arc3", color='gray'),
                bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.8))
    
plt.xlabel('\nLatency (ms) / Normalized latency (ms)*', fontsize=18)
plt.ylabel('Average Precision (%)', fontsize=12)
plt.title('TensorRT models: Unnormalized and GPU-normalized Latency vs. Average Precision\n', fontsize=18)
plt.grid(True, linestyle='--', alpha=0.5)

# Create legend
blue_patch = plt.Line2D([0], [0], marker='o', color='w', label='BEVDetNet TRT', 
                       markerfacecolor='blue', markersize=8)
red_patch = plt.Line2D([0], [0], marker='o', color='w', label='Our model TRT', 
                      markerfacecolor='red', markersize=8)
norm_patch = plt.Line2D([0], [0], marker='D', color='gray', linestyle='None', label='TRT normalized', 
                        markersize=8, alpha=0.8)
unnorm_patch = plt.Line2D([0], [0], marker='o', color='gray', linestyle='None', label='TRT unnormalized',
                        markersize=8, alpha=0.5)
size_legends = [plt.scatter([], [], c='gray', alpha=0.5, s=size*5, label=label) for size, label in zip([20, 40, 60], ['Small', 'Medium', 'Large'])]

# Size legend
size_legends = []
for size, label in zip([20, 40, 60], ['Small', 'Medium', 'Large']):
    size_legends.append(plt.scatter([], [], c='gray', alpha=0.5, s=size*5, label=label))

# Place legend further to the right
legend = plt.legend(handles=[blue_patch, red_patch, norm_patch, unnorm_patch] + size_legends,
                   title='Legend', loc='upper left', 
                   bbox_to_anchor=(1.01, 1),
                   borderaxespad=0.,
                   fontsize=13,
                   title_fontsize=15,
                   handletextpad=1.5,
                   labelspacing=1.2)

# Frame around the legend
legend.get_frame().set_alpha(0.8)
legend.get_frame().set_edgecolor('black')

# Add footnote
plt.figtext(0.1, 0.02, '*For a fair comparison, the latency of our GPU is scaled (ms * 29.77 TFLOPS / 11.34 TFLOPS ) to respective precision FLOPS values of GTX 1080Ti.', 
            fontsize=12, style='italic')

# Adjusting the layout for more space on the right and bottom
plt.tight_layout(rect=[0, 0.05, 0.82, 1])
plt.savefig('tensorrt_comparison_final.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()


# Issuance of standardization factors for control purposes
print("Standardized latency (ms * TFLOPS / TFLOPS):")
for i, model in enumerate(models):
    if i < 3:
        norm_latency = latency_original[i] / bevdetnet_flops
        gpu_info = f"{latency_original[i]}ms / {bevdetnet_flops} TFLOPS"
    else:
        if i == 3:
            norm_latency = latency_original[i] * rtx3080_fp32_flops / bevdetnet_flops
            gpu_info = f"{latency_original[i]}ms * {rtx3080_fp32_flops} / {bevdetnet_flops} TFLOPS"
        elif i == 4:
            norm_latency = latency_original[i] * rtx3080_fp16_flops / bevdetnet_flops
            gpu_info = f"{latency_original[i]}ms * {rtx3080_fp16_flops} / {bevdetnet_flops} TFLOPS"
        else:
            norm_latency = latency_original[i] * rtx3080_int8_flops / bevdetnet_flops
            gpu_info = f"{latency_original[i]}ms * {rtx3080_int8_flops} / {bevdetnet_flops} TFLOPS"
    print(f"{model}: {gpu_info} = {norm_latency:.3f} ms/TFLOPS")

## Paper: Average precision and computation latency for birds-eye-view detection measured on KITTI’s validation set. All models are in FP32 resolution to enable a fair comparison.

In [ ]:
models = [
    {"name": "MV3D", "latency_ms": 240, "tflops": 6.691},
    {"name": "vehicle", "latency_ms": 1, "tflops": 6.691},
    {"name": "BirdNet", "latency_ms": 110, "tflops": 1},
    {"name": "Complex-YOLO", "latency_ms": 19.8, "tflops": 12.15},
    {"name": "RT3D", "latency_ms": 89, "tflops": 6.691 },
    {"name": "PIXOR", "latency_ms": 35, "tflops": 12.15},
    {"name": "PIXOR++", "latency_ms": 35, "tflops": 1},
    {"name": "Complexer-YOLO", "latency_ms": 64.1, "tflops": 11.34},
    {"name": "HDNet", "latency_ms": 50, "tflops": 1},
    {"name": "BirdNet+", "latency_ms": 115, "tflops": 12.15},
    {"name": "BEVDetNet", "latency_ms": 3, "tflops": 6.447},
    {"name": "BEV-Net", "latency_ms": 37.1, "tflops": 15.67},
    {"name": "Ours", "latency_ms": 2.8, "tflops": 29.77},
]

print("Model\t\t\tLatency (ms)\tTFLOPS\tNormalized Latency (ms*TFLOPS)")
for m in models:
    norm_latency = m["latency_ms"] * m["tflops"]
    print(f"{m['name']:<20}\t{m['latency_ms']:<12}\t{m['tflops']:<7}\t{norm_latency:.3f}")

## Claude diagram script


In [7]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Image, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
import io

fig, ax = plt.subplots(figsize=(9, 5.5))

our_norm = {'x': [7.351], 'y': [86.03], 'labels': ['FP32 (49 MB)\n7.351 ms | 86.03%']}
our_unnorm = {'x': [2.8, 1.7, 1.5], 'y': [86.03, 86.01, 80.67], 'labels': ['FP32 (49 MB)\n2.8 ms | 86.03%', 'FP16 (25 MB)\n1.7 ms | 86.01%', 'INT8 (16 MB)\n1.5 ms | 80.67%']}
bev_desktop = {'x': [3], 'y': [87.51], 'labels': ['FP32 (59 MB)\n3.0 ms | 87.51%']}
bev_embedded = {'x': [6, 4, 2.1], 'y': [87.51, 86.44, 84.20], 'labels': ['FP32 (59 MB)\n6.0 ms | 87.51%', 'FP16 (39 MB)\n4.0 ms | 86.44%', 'INT8 (22 MB)\n2.1 ms | 84.20%']}

blue = '#185FA5'
coral = '#E24B4A'

ax.scatter(our_norm['x'], our_norm['y'], color=blue, marker='o', s=100, zorder=5, label='Our model TRT (normalized)')
ax.scatter(our_unnorm['x'], our_unnorm['y'], facecolors='none', edgecolors=blue, marker='o', s=100, linewidths=2, zorder=5, label='Our model TRT (unnormalized)')
ax.scatter(bev_desktop['x'], bev_desktop['y'], color=coral, marker='s', s=100, zorder=5, label='BEVDetNet TRT (RTX2080 Max-Q GPU)')
ax.scatter(bev_embedded['x'], bev_embedded['y'], facecolors='none', edgecolors=coral, marker='^', s=120, linewidths=2, zorder=5, label='BEVDetNet TRT (AGX Xavier GPU)')

label_offsets = {
    (7.351, 86.03): (10, 5),
    (2.8,  86.03):  (-2, 12),
    (1.7,  86.01):  (-10, 5),
    (1.5,  80.67):  (10, 5),
    (3.0,  87.51):  (10, 5),
    (6.0,  87.51):  (10, -20),
    (4.0,  86.44):  (10, 5),
    (2.1,  84.20):  (10, 5),
}
label_ha = {
    (1.7, 86.01): 'right',
}

for pts, lbls in [(our_norm, our_norm['labels']),
                  (our_unnorm, our_unnorm['labels']),
                  (bev_desktop, bev_desktop['labels']),
                  (bev_embedded, bev_embedded['labels'])]:
    for x, y, lbl in zip(pts['x'], pts['y'], lbls):
        xytext = label_offsets.get((x, y), (10, 5))
        ha = label_ha.get((x, y), 'left')
        ax.annotate(lbl, (x, y), textcoords='offset points', xytext=xytext,
                    fontsize=7.5, color='#5F5E5A', ha=ha)

ax.set_xlabel('Latency (ms)', fontsize=11, color='#444441')
ax.set_ylabel('Average Precision (%)', fontsize=11, color='#444441')
ax.set_xlim(0, 10)
ax.set_ylim(78, 90)
ax.tick_params(colors='#888780', labelsize=9)
ax.grid(True, color='#DDDDDD', linewidth=0.5, linestyle='--')
for spine in ax.spines.values():
    spine.set_edgecolor('#CCCCCC')

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=blue, markersize=9, label='Our model TRT (normalized)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none', markeredgecolor=blue, markeredgewidth=2, markersize=9, label='Our model TRT (unnormalized)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=coral, markersize=9, label='BEVDetNet TRT (RTX2080 Max-Q GPU)'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='none', markeredgecolor=coral, markeredgewidth=2, markersize=9, label='BEVDetNet TRT (AGX Xavier GPU)'),
]
ax.legend(handles=legend_elements, fontsize=8.5, framealpha=0.9, edgecolor='#CCCCCC', loc='lower right')

plt.tight_layout()

img_buffer = io.BytesIO()
plt.savefig(img_buffer, format='PNG', dpi=180, bbox_inches='tight')
img_buffer.seek(0)
plt.close()

output_path = 'latency_vs_precision.pdf'
doc = SimpleDocTemplate(output_path, pagesize=A4,
                        leftMargin=2*cm, rightMargin=2*cm,
                        topMargin=2.5*cm, bottomMargin=2*cm)

styles = getSampleStyleSheet()
title_style = ParagraphStyle('title', parent=styles['Heading2'],
                              fontSize=13, leading=18, spaceAfter=10,
                              textColor=colors.HexColor('#2C2C2A'))
caption_style = ParagraphStyle('caption', parent=styles['Normal'],
                                fontSize=9, leading=13, textColor=colors.HexColor('#5F5E5A'),
                                spaceBefore=8)

story = []
story.append(Paragraph(
    'TensorRT Models: Unnormalized and GPU-Normalized Latency vs. Average Precision',
    title_style))
story.append(Spacer(1, 0.3*cm))

img = Image(img_buffer, width=16*cm, height=9.5*cm)
story.append(img)

story.append(Paragraph(
    '<i>*For a fair comparison, the latency of our GPU is scaled (ms \u00d7 29.77 TFLOPS / 11.34 TFLOPS) '
    'to respective precision FLOPS values of GTX 1080 Ti.</i>',
    caption_style))

doc.build(story)
print("PDF saved to", output_path)


PDF saved to latency_vs_precision.pdf
